In [ ]:
%pip install -q -U kaggle

In [ ]:
from pathlib import Path
import shutil
import psutil

RAW = Path("/content/amex/raw")
RAW.mkdir(parents=True, exist_ok=True)

gb = 1024 ** 3

print(f"사용 가능한 RAM: {psutil.virtual_memory().available / gb:.1f} GB")
print(f"남은 디스크 공간: {shutil.disk_usage('/content').free / gb:.1f} GB")

In [ ]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle API 토큰 입력: ")

In [ ]:
!kaggle competitions files amex-default-prediction

In [ ]:
import subprocess

def download_file(filename):
    subprocess.run(
        [
            "kaggle", "competitions", "download",
            "amex-default-prediction",
            "-f", filename,
            "-p", str(RAW),
        ],
        check=True,
    )

download_file("train_labels.csv")

In [ ]:
def data_path(filename):
    for path in [RAW / filename, RAW / f"{filename}.zip"]:
        if path.exists():
            return path
    raise FileNotFoundError(f"{filename} 다운로드를 확인하세요.")

In [ ]:
import pandas as pd

labels = pd.read_csv(
    data_path("train_labels.csv"),
    dtype={"customer_ID": "string", "target": "int8"},
)

assert labels["customer_ID"].is_unique, "고객 ID 중복 확인 필요"
assert labels["target"].isin([0, 1]).all(), "정답 값 확인 필요"

print(f"고객 수: {len(labels):,}")
display(labels.head())

display(
    labels["target"]
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .to_frame("고객 수")
    .assign(비율=lambda x: x["고객 수"] / len(labels))
)

In [ ]:
download_file("train_data.csv")

In [ ]:
for path in sorted(RAW.iterdir()):
    print(f"{path.name}: {path.stat().st_size / gb:.2f} GB")

print(f"\n남은 디스크: {shutil.disk_usage('/content').free / gb:.1f} GB")

In [ ]:
preview = pd.read_csv(
    data_path("train_data.csv"),
    nrows=1_000,
    dtype={"customer_ID": "string"},
    parse_dates=["S_2"],
)

print("미리보기 크기:", preview.shape)
print("미리보기의 고객 수:", preview["customer_ID"].nunique())

display(preview.head())
preview.info()

In [ ]:
CAT_COLS = [
    "B_30", "B_38", "D_114", "D_116", "D_117",
    "D_120", "D_126", "D_63", "D_64", "D_66", "D_68",
]

print("범주형 변수:", CAT_COLS)

display(
    preview.isna()
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("미리보기 결측 비율")
)

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
N_CUSTOMERS = 30_000

sample_labels, _ = train_test_split(
    labels,
    train_size=N_CUSTOMERS,
    stratify=labels["target"],
    random_state=SEED,
)

sample_labels = sample_labels.reset_index(drop=True)
selected_ids = set(sample_labels["customer_ID"])

print(f"선택한 고객 수: {len(selected_ids):,}")

display(
    sample_labels["target"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("표본 내 비율")
    .to_frame()
)

In [ ]:
CAT_COLS = [
    "B_30", "B_38", "D_114", "D_116", "D_117",
    "D_120", "D_126", "D_63", "D_64", "D_66", "D_68",
]

NUM_COLS = [
    col for col in preview.columns
    if col not in ["customer_ID", "S_2"] + CAT_COLS
]

dtype_map = {col: "float32" for col in NUM_COLS}
dtype_map.update({col: "string" for col in CAT_COLS})
dtype_map["customer_ID"] = "string"

print(f"수치형 변수: {len(NUM_COLS)}개")
print(f"범주형 변수: {len(CAT_COLS)}개")

In [ ]:
import time
import pandas as pd

started = time.perf_counter()
parts = []
rows_scanned = 0
rows_selected = 0

with pd.read_csv(
    data_path("train_data.csv"),
    dtype=dtype_map,
    parse_dates=["S_2"],
    chunksize=50_000,
) as reader:

    for i, chunk in enumerate(reader, start=1):
        matched = chunk.loc[
            chunk["customer_ID"].isin(selected_ids)
        ].copy()

        if not matched.empty:
            parts.append(matched)

        rows_scanned += len(chunk)
        rows_selected += len(matched)

        if i == 1 or i % 10 == 0:
            elapsed = (time.perf_counter() - started) / 60
            print(
                f"읽은 행: {rows_scanned:,} | "
                f"추출한 행: {rows_selected:,} | "
                f"경과: {elapsed:.1f}분",
                flush=True,
            )

if not parts:
    raise ValueError("추출된 데이터가 없습니다. 고객 ID를 확인하세요.")

sample = pd.concat(parts, ignore_index=True)
del parts, chunk, matched

sample = sample.sort_values(
    ["customer_ID", "S_2"],
    ignore_index=True,
)

elapsed_seconds = time.perf_counter() - started

print(f"\n완료: {elapsed_seconds / 60:.1f}분")
print("표본 크기:", sample.shape)
print(
    "표본 메모리:",
    f"{sample.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
)

In [ ]:
assert set(sample["customer_ID"]) == selected_ids, \
    "선택한 고객 중 이력이 없는 고객이 있습니다."

assert sample["S_2"].notna().all(), \
    "날짜가 비어 있는 행이 있습니다."

duplicate_count = sample.duplicated(
    ["customer_ID", "S_2"]
).sum()

print(f"고객 수: {sample['customer_ID'].nunique():,}")
print(f"전체 기록 수: {len(sample):,}")
print(f"고객·날짜 중복: {duplicate_count:,}")
print(f"날짜 범위: {sample['S_2'].min()} ~ {sample['S_2'].max()}")

print("\n고객별 기록 수")
display(
    sample.groupby("customer_ID")
    .size()
    .describe()
    .to_frame("기록 수")
)

assert duplicate_count == 0, \
    "고객·날짜 중복 원인을 확인한 뒤 진행하세요."

In [ ]:
missing_report = (
    sample[NUM_COLS + CAT_COLS]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("결측률")
    .to_frame()
)

display(missing_report.head(20))

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/amex_project")
PROCESSED = PROJECT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

sample.to_parquet(
    PROCESSED / "customer_history_30k.parquet",
    index=False,
)

sample_labels.to_parquet(
    PROCESSED / "customer_labels_30k.parquet",
    index=False,
)

missing_report.to_csv(
    PROCESSED / "missing_report_30k.csv"
)

for path in sorted(PROCESSED.iterdir()):
    print(f"{path.name}: {path.stat().st_size / 1024**2:.1f} MB")

# Baseline (Light GBM)